- Analyse the location data we have for applications for deals
    - E.g., for physical deals, can we get an estimate of the average distance of the creator to the deal? Does it matter? How can I tie this back to the model predicting apps? 

# Init

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
import pandas as pd

from src.utils import interactive_dataframe_selector

In [2]:
# Load environment variables from .env file
load_dotenv("../private_data/.env")

host = os.getenv("HOST")
db = os.getenv("DB")
port = os.getenv("PORT")
role = os.getenv("ROLE")
pw = os.getenv("PASSWORD")
engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

ValueError: invalid literal for int() with base 10: 'None'

# Creators

In [ ]:
query = """
SELECT 
    id, 
    user_id,
    name, 
    rating,
    description,
    status,
    created_at,
    CAST(updated_at AS TEXT) as updated_at,
    CAST(birthday AS TEXT) as birthday,
    gender,
    reviews_count
FROM public.influencers;
"""

with engine.connect() as connection:
    # 2. We don't use parse_dates=True here to avoid the crash
    df_creators = pd.read_sql(text(query), connection)

query = """
SELECT 
    created_at,
    updated_at,
    influencer_id,
    name,
    country,
    latitude,
    longitude
FROM public.influencer_locations;
"""


with engine.connect() as connection:
    # 2. We don't use parse_dates=True here to avoid the crash
    df_creatorsloc = pd.read_sql(text(query), connection)


# Strip the timezone from both (just to be safe)
df_creators['created_at'] = pd.to_datetime(df_creators['created_at']).dt.tz_localize(None)
df_creatorsloc['created_at'] = pd.to_datetime(df_creatorsloc['created_at']).dt.tz_localize(None)

# Now the comparison will work perfectly
mask = df_creators['created_at'] < df_creatorsloc['created_at'].min()
df_creatorsloc